# Lab 5: Reliable Order Processing System

**Difficulty: Advanced | ~50 min | Requires Labs 1–4**

*Lab 5 of 7 in the MongoDB Mastery series.*

In this lab, you will build a reliable order processing system with atomic operations, revenue analytics, change stream simulation, and backup/restore.

You will learn how to:
1. Use `find_one_and_update` for atomic order placement and cancellation
2. Prevent overselling with atomic guard conditions
3. Build a revenue aggregation pipeline with `$lookup`
4. Simulate change streams for real-time order monitoring
5. Perform backup and restore operations

In [ ]:
!pip install -qU pymongo==4.10.1 mongomock

This installs `pymongo` (the MongoDB driver) and `mongomock` (in-memory mock server) so you can practice without installing MongoDB.

### Step 1 — Connect and Create Collections

In [ ]:
import pymongo
import mongomock
import json
from datetime import datetime

client = mongomock.MongoClient()
db = client["order_system"]
inventory = db["inventory"]
orders = db["orders"]
customers = db["customers"]

print("Connected to order_system database")

We create three collections: `inventory` for product stock levels, `orders` for order records, and `customers` for registered users. This three-collection design separates concerns — stock management, order processing, and user data are independent domains.

### Step 2 — Seed Inventory

In [ ]:
inventory_data = [
    {"product_id": "P001", "name": "Wireless Mouse",  "price": 29.99, "quantity": 20},
    {"product_id": "P002", "name": "Keyboard",        "price": 49.99, "quantity": 15},
    {"product_id": "P003", "name": "USB Hub",         "price": 24.99, "quantity": 30},
    {"product_id": "P004", "name": "Monitor",         "price": 249.99, "quantity": 8},
    {"product_id": "P005", "name": "Laptop Stand",    "price": 49.99, "quantity": 15},
]

inventory.insert_many(inventory_data)
print(f"Inventory: {inventory.count_documents({})} products")
for item in inventory.find({}, {"_id": 0}):
    print(f"  {item['name']:<18} | ${item['price']:<7} | qty: {item['quantity']}")

Each inventory document tracks `quantity` — the number of units in stock. This field will be atomically decremented when orders are placed and incremented when orders are cancelled.

### Step 3 — Seed Order History and Customers

In [ ]:
customer_data = [
    {"customer_id": "alice",   "name": "Alice Johnson",  "email": "alice@example.com"},
    {"customer_id": "bob",     "name": "Bob Smith",     "email": "bob@example.com"},
    {"customer_id": "charlie", "name": "Charlie Brown", "email": "charlie@example.com"},
]
customers.insert_many(customer_data)

order_data = [
    {"order_id": "ORD-1001", "customer_id": "alice",   "product_id": "P001", "quantity": 2, "total": 59.98,  "date": "2025-01-15", "status": "completed"},
    {"order_id": "ORD-1002", "customer_id": "bob",     "product_id": "P002", "quantity": 1, "total": 49.99,  "date": "2025-02-20", "status": "completed"},
    {"order_id": "ORD-1003", "customer_id": "charlie", "product_id": "P004", "quantity": 1, "total": 249.99, "date": "2025-03-10", "status": "completed"},
    {"order_id": "ORD-1004", "customer_id": "alice",   "product_id": "P003", "quantity": 1, "total": 24.99,  "date": "2025-04-05", "status": "completed"},
    {"order_id": "ORD-1005", "customer_id": "bob",     "product_id": "P002", "quantity": 2, "total": 99.98,  "date": "2025-05-12", "status": "completed"},
]
orders.insert_many(order_data)

print(f"Customers: {customers.count_documents({})}")
print(f"Orders:    {orders.count_documents({})}")

The order history provides data for the aggregation pipeline in Step 7. Each order tracks `status` (completed, pending, cancelled) to support different order lifecycle operations.

### Step 4 — Atomic Order Placement (find_one_and_update)

In [ ]:
def place_order(customer_id, product_id, qty):
    """Place an order atomically: decrement stock + create order in one step."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    order_id = f"ORD-S{orders.count_documents({}) + 1001}"

    updated = inventory.find_one_and_update(
        {"product_id": product_id, "quantity": {"$gte": qty}},
        {"$inc": {"quantity": -qty}},
    )

    if updated is None:
        item = inventory.find_one({"product_id": product_id}, {"_id": 0})
        if item and item["quantity"] < qty:
            return None, f"Only {item['quantity']} {item['name']}(s) in stock, order was for {qty}"
        return None, f"Product {product_id} not found"

    total = updated["price"] * qty
    orders.insert_one({
        "order_id": order_id, "customer_id": customer_id,
        "product_id": product_id, "quantity": qty,
        "total": total, "date": now, "status": "completed",
    })

    updated_stock = inventory.find_one({"product_id": product_id}, {"_id": 0, "quantity": 1})
    return order_id, f"${total:.2f} | Stock remaining: {updated_stock['quantity']}"

order_id, result = place_order("alice", "P005", 1)
print(f"Order placed: {order_id} | Laptop Stand x1 | {result}")

The key design decision: `find_one_and_update` with `{"$gte": qty}` as the filter is a single atomic operation. The filter checks stock *and* decrements in one step — no window where another process could modify the quantity between the check and the decrement. This prevents overselling under concurrent access.

### Step 5 — Atomic Order Cancellation

In [ ]:
def cancel_order(order_id):
    """Cancel an order atomically: restore stock + remove order in one step."""
    order = orders.find_one({"order_id": order_id})
    if not order:
        return False, "Order not found"

    inventory.update_one(
        {"product_id": order["product_id"]},
        {"$inc": {"quantity": order["quantity"]}}
    )
    orders.delete_one({"order_id": order_id})
    return True, f"Restored {order['quantity']} unit(s) of {order['product_id']}"

success, msg = cancel_order("ORD-S1001")
print(f"Cancellation: {msg}")

item = inventory.find_one({"product_id": "P005"}, {"_id": 0})
print(f"Laptop Stand stock after cancel: {item['quantity']}")

Cancellation reverses the stock change atomically using `$inc` with a positive value. In a production system, you would wrap both the `$inc` and `delete_one` in a multi-document transaction to ensure both succeed or both fail. With `mongomock`, we rely on the single-document atomic `$inc` for the critical stock update.

### Step 6 — Insufficient Stock (Failure Handling)

In [ ]:
order_id, msg = place_order("charlie", "P005", 20)
print(f"Failed: {msg}")

item = inventory.find_one({"product_id": "P005"}, {"_id": 0})
print(f"Stock unchanged: {item['name']} quantity = {item['quantity']}")

When the filter `{"$gte": 20}` fails (stock is less than 20), `find_one_and_update` returns `None` and makes no changes. This is the atomic guard in action — the stock check and decrement are a single operation, so there is no partial state where stock was decremented but no order was created.

### Step 7 — Revenue by Product (Aggregation)

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "inventory", "localField": "product_id",
        "foreignField": "product_id", "as": "product"
    }},
    {"$unwind": "$product"},
    {"$group": {
        "_id": "$product.name",
        "total_qty": {"$sum": "$quantity"},
        "total_revenue": {"$sum": "$total"}
    }},
    {"$sort": {"total_revenue": -1}},
    {"$project": {"_id": 0, "product": "$_id", "total_qty": 1,
                  "total_revenue": {"$round": ["$total_revenue", 2]}}}
]

print("--- Revenue by Product ---")
for doc in orders.aggregate(pipeline):
    print(f"{doc['product']:<18} | {doc['total_qty']} sold | ${doc['total_revenue']}")

This pipeline joins orders with inventory to get product names, groups by product to sum quantities and revenue, and sorts by revenue descending. The `$round` stage ensures clean currency formatting in the output.

### Step 8 — Change Stream Simulation

In [ ]:
new_order_id, _ = place_order("bob", "P001", 1)

change_events = []
for order in orders.find({"order_id": {"$regex": "^ORD-S"}}):
    product = inventory.find_one({"product_id": order["product_id"]}, {"_id": 0, "name": 1})
    change_events.append({
        "operationType": "insert",
        "fullDocument": {
            "order_id": order["order_id"],
            "customer_id": order["customer_id"],
            "product": product["name"],
            "quantity": order["quantity"],
            "total": order["total"],
        }
    })

print("--- Change Stream Events (Simulated) ---")
for evt in change_events:
    doc = evt["fullDocument"]
    print(f"[{evt['operationType'].upper()}] order_id={doc['order_id']} "
          f"customer={doc['customer_id']} product={doc['product']} "
          f"qty={doc['quantity']} total=${doc['total']:.2f}")

In production MongoDB, `orders.watch()` returns a cursor that yields change events as documents are inserted or updated. Since `mongomock` does not support native change streams, we simulate the pattern by querying for newly inserted orders and constructing event objects. The structure mirrors what a real change stream event looks like.

### Step 9 — Backup and Restore

In [ ]:
backup = {}
for name in ["orders", "inventory", "customers"]:
    backup[name] = list(db[name].find({}, {"_id": 0}))

with open("order_system_backup.json", "w") as f:
    json.dump(backup, f, indent=2)

print(f"Backup: {len(backup['orders'])} orders, "
      f"{len(backup['inventory'])} inventory items, "
      f"{len(backup['customers'])} customers exported")

The backup exports each collection to a JSON file. In production, you would use `mongodump` for binary backups (faster, supports point-in-time recovery) or MongoDB Atlas continuous backups for managed cloud deployments.

In [ ]:
for name in ["orders", "inventory", "customers"]:
    db[name].drop()
    db[name] = db.create_collection(name)

with open("order_system_backup.json", "r") as f:
    backup = json.load(f)

for name in ["orders", "inventory", "customers"]:
    if backup[name]:
        db[name].insert_many(backup[name])

print(f"Restore: {db['orders'].count_documents({})} orders, "
      f"{db['inventory'].count_documents({})} inventory items, "
      f"{db['customers'].count_documents({})} customers restored")

Restore clears each collection and re-imports from the JSON backup. The `drop()` + `create_collection()` pattern ensures a clean slate — no stale documents remain from the original data. In production, `mongorestore` handles this with journaling and corruption detection.

### Step 10 — Create Indexes

In [ ]:
orders.create_index("customer_id")
orders.create_index("date")
orders.create_index([("customer_id", 1), ("date", -1)])

print("Indexes created.")
print("Orders indexes:", list(orders.index_information().keys()))

The compound index on `(customer_id, date)` supports the common query pattern "show me this customer's orders, most recent first" — the index covers both the filter and the sort, avoiding an in-memory sort stage.

### Step 11 — Print Summary Report

In [ ]:
total_orders = orders.count_documents({})
total_inventory = inventory.count_documents({})
total_customers = customers.count_documents({})

rev_pipeline = [
    {"$group": {"_id": None, "total": {"$sum": "$total"}}}
]
total_revenue = list(orders.aggregate(rev_pipeline))[0]["total"]

print("       RELIABLE ORDER PROCESSING — SUMMARY REPORT")
print(f"\nOrders:    {total_orders}")
print(f"Inventory: {total_inventory} products")
print(f"Customers: {total_customers}")
print(f"Revenue:   ${total_revenue:.2f}")
print(f"\n--- Indexes ---")
print(f"  Orders: {list(orders.index_information().keys())}")